# Worked block analysis — CAN transceiver

This notebook is the **smallest thing that looks like a real block analysis** with only steps 1 + 2 of the framework in place (Quantity + Pint + scenario/mode TOML loaders). No Hamilton DAG, no Contracts, no VerificationTests — those land in steps 4, 7, and 9.

**The block.** A CAN bus transceiver on a 5 V rail. Three quantities of interest:

1. **Supply power** — current draw varies by mode (`off` / `sleep` / `active` / `diagnostic`); supply voltage varies by tolerance (±5 % of 5 V).
2. **Junction temperature** — `ambient_temp + dissipation × R_θJA`. Ambient comes from the project scenarios.
3. **Spec checks** — power ≤ 500 mW and T_J ≤ 125 °C across every (scenario × mode) combination.

**Mental model.** Think of the analysis as a pipe:

```
scenarios.toml  ─┐
modes.toml     ─┤
leaf inputs    ─┴─▶  Quantity arithmetic  ─▶  derived Quantities  ─▶  spec checks
```

Every derived value carries its own (scenario × mode) span; the framework just propagates.

## 1. Load project context

Scenarios = environmental/operating corners (cold-low-Vbat, nominal, hot-high-Vbat).
Modes = system states (off, sleep, active, diagnostic).

In [1]:
from pathlib import Path

from framework import (
    Constant, Quantity, RangeQuantity,
    load_scenarios, load_modes,
    requirements,
)
from framework.units import V, A, mA, uA, W, mW, K, degC, Ohm, Hz

PROJECT = Path.cwd() / "project"
scenarios = load_scenarios(PROJECT / "scenarios.toml")
modes     = load_modes(PROJECT / "modes.toml")

# Importing project.requirements is what populates framework.requirements
# (each declaration auto-registers by its req= ID).
from project.requirements import (
    OPERATING_TEMP,
    CAN_5V_RAIL,
    VBAT,
    TOTAL_QUIESCENT_CURRENT_BUDGET,
)

print(f"Loaded {len(scenarios)} scenarios:    {scenarios.names()}")
print(f"Loaded {len(modes)} modes:        {modes.names()}")
print(f"Registered {len(requirements.list_all())} requirements:")
for r in requirements.list_all():
    print(f"  {r.req:15s}  {type(r).__name__:18s}  {r.description}")

Loaded 3 scenarios:    ['nominal', 'cold_low_vin', 'hot_high_vin']
Loaded 4 modes:        ['off', 'sleep', 'active', 'diagnostic']
Registered 5 requirements:
  REQ-ENV-001      TempRange           Vehicle operating ambient temperature envelope (key-on)
  REQ-ENV-002      TempRange           Vehicle storage temperature envelope (long-term, powered-down)
  REQ-PWR-001      SupplyEnvelope      Vehicle 12V system with cold-crank and load-dump margins
  REQ-PWR-005      SupplyEnvelope      5V analog/peripheral rail (steady-state +/-5%) feeding the CAN transceiver and the ADC reference
  REQ-PWR-014      CurrentBudget       Vehicle key-off total quiescent current budget across all blocks


## 2. Block leaf inputs

*"Leaves" are the hand-coded inputs the engineer owns: datasheet values, design choices, mode-dependent currents. They're the entry points of the DAG.* (Design doc 6.6 — block `leaves.py`.)

For the CAN transceiver we declare:

- `i_supply` — mode-dependent supply current, with a min/max range per mode for component tolerance.
- `v_supply` — 5 V ±5 % rail tolerance (scenario-invariant in this simple version; in a real project this would come from the power-supply block's Contract).
- `r_theta_ja` — thermal resistance junction-to-ambient (SO-8 in still air).
- `t_j_max_spec` — junction-temp limit from the datasheet (125 °C derate of 150 °C absolute max).

In [2]:
# Mode-dependent supply current — every value is a RangeQuantity to capture
# part-to-part variation pulled from the datasheet min/max columns.
i_supply = Quantity(unit=A, by_mode={
    "off":        Constant(0.0, A),
    "sleep":      RangeQuantity(8e-6,  15e-6, A),
    "active":     RangeQuantity(45e-3, 65e-3, A),
    "diagnostic": RangeQuantity(70e-3, 90e-3, A),
})

# The 5 V rail comes from project requirement REQ-PWR-005 (CAN_5V_RAIL).
# In a future iteration this becomes a Contract import from blocks/power_supply
# whose Contract is itself bounded by CAN_5V_RAIL.
v_supply   = RangeQuantity(CAN_5V_RAIL.min, CAN_5V_RAIL.max)

r_theta_ja = Constant(120.0, K / W)         # SO-8, still air (datasheet)
t_j_max    = Constant(125.0, degC)          # 25 C derate from datasheet T_J_max = 150 C

## 3. Pull ambient temperature from the project scenarios

Every scenario carries `ambient_temp` in its context dict. `ScenarioSet.as_quantity(...)` materializes a `Quantity(by_scenario=...)` from a single key across the set. We pull ambient straight into Kelvin so the thermal math downstream stays multiplicative (Pint refuses to add a K-unit delta to a degC-unit absolute, by design).

In [3]:
ambient_temp = scenarios.as_quantity("ambient_temp", unit=K)

print("Ambient by scenario:")
for s in scenarios.names():
    t_k = ambient_temp.at(scenario=s)
    t_c = t_k - 273.15
    print(f"  {s:14s} {t_k:7.2f} K  ({t_c:+5.1f} °C)")

Ambient by scenario:
  nominal         298.15 K  (+25.0 °C)
  cold_low_vin    233.15 K  (-40.0 °C)
  hot_high_vin    358.15 K  (+85.0 °C)


## 4. Derive power dissipation = `V × I`

`v_supply` has no axes (just a scalar range). `i_supply` has a `by_mode` axis with a range per mode. The product carries the mode axis and the propagated range, with no by-hand bookkeeping.

In [4]:
power = (v_supply * i_supply).to(mW)

print(f"{'mode':12s} {'min':>10s} {'max':>10s}")
print("-" * 36)
for m in modes.names():
    val = power.at(mode=m)
    if isinstance(val, tuple):
        lo, hi = val
    else:
        lo = hi = val
    print(f"{m:12s} {lo:9.3f}mW {hi:9.3f}mW")

mode                min        max
------------------------------------
off              0.000mW     0.000mW
sleep            0.038mW     0.079mW
active         213.750mW   341.250mW
diagnostic     332.500mW   472.500mW


## 5. Derive junction temperature = `T_ambient + P × R_θJA`

`power` lives on the mode axis (with ranges); `ambient_temp` lives on the scenario axis; `r_theta_ja` is a scalar constant. Adding them produces a Quantity that varies along **both axes simultaneously** — every (scenario × mode) combination gets its own min/max.

In [5]:
thermal_rise = (power * r_theta_ja).to(K)
t_j_kelvin   = ambient_temp + thermal_rise
t_j          = t_j_kelvin.to(degC)

def cell_str(val):
    if isinstance(val, tuple):
        lo, hi = val
        return f"{lo:5.1f} – {hi:5.1f} °C"
    return f"{val:5.1f} °C"

scen_names = scenarios.names()
mode_names = modes.names()
print(f"{'mode':12s} | " + " | ".join(f"{s:^16s}" for s in scen_names))
print("-" * (14 + 19 * len(scen_names)))
for m in mode_names:
    cells = [cell_str(t_j.at(mode=m, scenario=s)) for s in scen_names]
    print(f"{m:12s} | " + " | ".join(f"{c:^16s}" for c in cells))

mode         |     nominal      |   cold_low_vin   |   hot_high_vin  
-----------------------------------------------------------------------
off          |      25.0 °C     |     -40.0 °C     |      85.0 °C    
sleep        |  25.0 –  25.0 °C | -40.0 – -40.0 °C |  85.0 –  85.0 °C
active       |  50.6 –  65.9 °C | -14.4 –   0.9 °C | 110.6 – 125.9 °C
diagnostic   |  64.9 –  81.7 °C |  -0.1 –  16.7 °C | 124.9 – 141.7 °C


## 6. Spec checks

`.within(lo, hi)` returns `True` iff **every** scenario/mode evaluation lies inside the spec. If you only need a yes/no for CI, this is your friend; for narrow margins or design-review reports we'll dig into the worst cell explicitly.

In [6]:
power_ok = power.within(0 * mW, 500 * mW)

# Lower T_J bound tied to project requirement REQ-ENV-001 (OPERATING_TEMP);
# upper bound is the block-local datasheet derate (t_j_max from leaves).
# within() accepts pint.Quantity, framework.Quantity, or plain floats.
t_j_ok = t_j.within(OPERATING_TEMP.min, t_j_max)

print(f"Power dissipation <= 500 mW everywhere?           {power_ok}")
print(f"Junction temp within [{OPERATING_TEMP.min:~P}, {t_j_max.at():.0f} C]?   {t_j_ok}")
print(f"  (linked to {OPERATING_TEMP.req} / block-local datasheet derate)")

# When something fails, surface the worst-case corner so the engineer
# knows where to look. Step 9 (VerificationTest) will do this automatically
# with margin-to-spec and a Jama link; here we just iterate.
if not t_j_ok:
    print("\nT_J failures:")
    for m in mode_names:
        for s in scen_names:
            val = t_j.at(mode=m, scenario=s)
            hi = val[1] if isinstance(val, tuple) else val
            if hi > 125.0:
                print(f"  mode={m:11s} scenario={s:14s} T_J max = {hi:5.1f} C   (over by {hi - 125:.1f} C)")

Power dissipation <= 500 mW everywhere?           True
Junction temp within [-40.0 °C, 125 C]?   False
  (linked to REQ-ENV-001 / block-local datasheet derate)

T_J failures:
  mode=active      scenario=hot_high_vin   T_J max = 125.9 C   (over by 0.9 C)
  mode=diagnostic  scenario=hot_high_vin   T_J max = 141.7 C   (over by 16.7 C)


## 7. What this would look like in the eventual full framework

Once steps 4–9 land, the **same analysis** moves into a block subpackage and becomes declarative:

```
blocks/can_transceiver/
  leaves.py         # i_supply, r_theta_ja, t_j_max   (the section 2 cell)
  analysis.py       # power(), t_j() as Hamilton DAG nodes
  contracts.py      # @contract def can_5v_draw() -> Quantity
  verifications.py  # @verification_test def test_t_j_below_spec(ctx) -> TestResult
  report.ipynb      # cell 1 runs results = project.run(block="can_transceiver")
                    # remaining cells are pure rendering of `results`
```

What changes:
- **Hamilton** wires the function signatures into a DAG, so adding a new derived Quantity is just writing a function with the right parameter names.
- **Contracts** make `can_5v_draw` consumable by `blocks/power_supply` without an import cycle.
- **VerificationTests** replace the manual `within(...)` block at the bottom — same logic, but reportable to Jama and the PR comment bot.
- **Caching** means re-running the notebook after editing a single leaf only recomputes what changed.

What does **not** change: the Quantity arithmetic above is exactly what the framework will do under Hamilton. The mental model is the right one.